# Sudoku-Algorithmen

Sudokus von sudoku.com.  
Die `0` repräsentiert ein leeres Feld.

- `sudoku` ist ein Sudoku aus der Kategorie **leicht** mit 44 freien Feldern
- `sudoku_diff` ist ein Sudoku aus der Kategorie **Extrem** mit 59 freien Feldern

Ignorieren wir die Arithmetik, wären das etwa 9^59 im Vergleich zu 9^44 Möglichkeiten.

## Imports

In [72]:
import copy
import time
import tracemalloc

## Sudoku-Ausgangszustände

In [73]:
sudoku = [
    [0, 0, 6, 0, 0, 0, 8, 0, 2],
    [7, 0, 0, 4, 2, 8, 0, 9, 6],
    [2, 1, 0, 0, 3, 0, 7, 0, 5],

    [0, 3, 1, 0, 0, 0, 9, 8, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 7],
    [8, 2, 0, 9, 5, 0, 0, 0, 3],

    [3, 0, 0, 0, 0, 2, 0, 6, 0],
    [0, 8, 5, 0, 7, 6, 0, 0, 0],
    [9, 0, 2, 5, 0, 1, 3, 0, 8]
]

sudoku_diff = [
    [0, 0, 6, 9, 0, 0, 0, 0, 0],
    [4, 0, 0, 0, 0, 0, 0, 0, 8],
    [0, 0, 0, 0, 5, 0, 0, 7, 0],

    [6, 0, 0, 0, 0, 4, 0, 0, 0],
    [1, 0, 8, 6, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 9, 5, 2, 0],

    [0, 0, 5, 3, 0, 0, 0, 0, 0],
    [0, 9, 0, 0, 0, 0, 8, 0, 0],
    [2, 4, 3, 0, 6, 0, 0, 0, 0]
]

## Hilfsfunktionen

Diese Funktionen werden von allen Algorithmen gemeinsam genutzt.

In [74]:
def print_sudoku(board):
    """Gibt das Sudoku-Board formatiert auf der Konsole aus."""
    for a in range(9):
        if a % 3 == 0 and a != 0:
            print("-" * 21)
        for b in range(9):
            if b % 3 == 0 and b != 0:
                print("|", end=" ")
            print(board[a][b], end=" ")
        print()


def find_empty(board):
    """Gibt die Position des nächsten leeren Feldes zurück."""
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                return row, col
    return None


def is_valid(board, num, row, col):
    """Überprüft, ob num an der übergebenen Position gültig ist."""
    # Zeile prüfen
    for b in range(9):
        if board[row][b] == num and b != col:
            return False

    # Spalte prüfen
    for a in range(9):
        if board[a][col] == num and a != row:
            return False

    # Block prüfen
    start_row = (row // 3) * 3
    start_col = (col // 3) * 3
    for a in range(start_row, start_row + 3):
        for b in range(start_col, start_col + 3):
            if board[a][b] == num and (a, b) != (row, col):
                return False

    return True


def measure(func, *args):
    """Führt func(*args) aus und gibt (ergebnis, laufzeit_s, peak_speicher_kb) zurück."""
    tracemalloc.start()
    start = time.perf_counter()
    result = func(*args)
    end = time.perf_counter()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, end - start, peak / 1024

def find_empty_mrv(board):
    least_num = None
    least_num_count = 9
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                count = 0
                for num in range(1,10):
                    if is_valid(board, num, row, col):
                        count = count +1
                if least_num_count > count:
                    least_num = row, col
                    least_num_count = count
    return least_num

print_sudoku(sudoku)

0 0 6 | 0 0 0 | 8 0 2 
7 0 0 | 4 2 8 | 0 9 6 
2 1 0 | 0 3 0 | 7 0 5 
---------------------
0 3 1 | 0 0 0 | 9 8 0 
0 0 0 | 1 0 0 | 0 0 7 
8 2 0 | 9 5 0 | 0 0 3 
---------------------
3 0 0 | 0 0 2 | 0 6 0 
0 8 5 | 0 7 6 | 0 0 0 
9 0 2 | 5 0 1 | 3 0 8 


---

# Depth-First Search (DFS)

## DFS auf einfachem Sudoku

Hier braucht der DFS nur sehr wenig Zeit.

In [75]:
def solve_dfs(board):
    """Löst ein Sudoku per rekursivem DFS (Backtracking)."""
    empty = find_empty(board)

    # Kein leeres Feld mehr → Sudoku gelöst
    if empty is None:
        return board

    row, col = empty

    # Probiere Zahlen von 1 bis 9
    for num in range(1, 10):
        if is_valid(board, num, row, col):
            board[row][col] = num

            if solve_dfs(board):
                return board

            board[row][col] = 0  # Backtrack

    return False


board = copy.deepcopy(sudoku)
solved, laufzeit, speicher = measure(solve_dfs, board)

if solved:
    print("Gelöstes Sudoku (einfach):")
    print_sudoku(solved)
else:
    print("Keine Lösung gefunden")

print(f"Laufzeit: {laufzeit:.6f} s")
print(f"Speicher:  {speicher:.2f} KB")

Gelöstes Sudoku (einfach):
4 9 6 | 7 1 5 | 8 3 2 
7 5 3 | 4 2 8 | 1 9 6 
2 1 8 | 6 3 9 | 7 4 5 
---------------------
5 3 1 | 2 6 7 | 9 8 4 
6 4 9 | 1 8 3 | 2 5 7 
8 2 7 | 9 5 4 | 6 1 3 
---------------------
3 7 4 | 8 9 2 | 5 6 1 
1 8 5 | 3 7 6 | 4 2 9 
9 6 2 | 5 4 1 | 3 7 8 
Laufzeit: 0.002937 s
Speicher:  6.94 KB


## DFS auf extremem Sudoku

Hier braucht der DFS deutlich länger, um auf das Ergebnis zu kommen, hat aber nur einen leicht erhöhten Speicherverbrauch. Das lässt sich mit der größeren Tiefe des Suchbaums durch die höhere Anzahl leerer Felder erklären.

In [76]:
board = copy.deepcopy(sudoku_diff)
solved, laufzeit, speicher = measure(solve_dfs, board)

if solved:
    print("Gelöstes Sudoku (extrem):")
    print_sudoku(solved)
else:
    print("Keine Lösung gefunden")

print(f"Laufzeit: {laufzeit:.6f} s")
print(f"Speicher:  {speicher:.2f} KB")

Gelöstes Sudoku (extrem):
5 8 6 | 9 4 7 | 3 1 2 
4 3 7 | 2 1 6 | 9 5 8 
9 1 2 | 8 5 3 | 6 7 4 
---------------------
6 2 9 | 5 3 4 | 7 8 1 
1 5 8 | 6 7 2 | 4 3 9 
3 7 4 | 1 8 9 | 5 2 6 
---------------------
8 6 5 | 3 9 1 | 2 4 7 
7 9 1 | 4 2 5 | 8 6 3 
2 4 3 | 7 6 8 | 1 9 5 
Laufzeit: 20.990699 s
Speicher:  7.30 KB


---

# DFS mit Tiefenbeschränkung

Der DFS mit Beschränkung ist bei der Lösung eines Sudokus wenig hilfreich, da die sinnvolle Beschränkung schon durch die Anzahl der freien Felder (AfF) vorgegeben ist:

- Ein Limit **< AfF** führt nie zu einem Ergebnis, da das Sudoku nicht vollständig ausgefüllt wird.
- Ein Limit **> AfF** bringt nichts, da die Suche nach der AfF ohnehin endet.

In [77]:
def solve_dfs_limited(board, depth, limit):
    """DFS mit maximaler Tiefenbeschränkung."""
    empty = find_empty(board)

    # Kein leeres Feld mehr → Sudoku gelöst
    if empty is None:
        return board
    if depth > limit:
        return False

    row, col = empty

    for num in range(1, 10):
        if is_valid(board, num, row, col):
            board[row][col] = num

            if solve_dfs_limited(board, depth + 1, limit):
                return board

            board[row][col] = 0  # Backtrack

    return False


LIMIT = 59  # Anzahl freier Felder in sudoku_diff

board = copy.deepcopy(sudoku_diff)
solved, laufzeit, speicher = measure(solve_dfs_limited, board, 0, LIMIT)

if solved:
    print("Gelöstes Sudoku (extrem, mit Limit):")
    print_sudoku(solved)
else:
    print("Keine Lösung gefunden")

print(f"Laufzeit: {laufzeit:.6f} s")
print(f"Speicher:  {speicher:.2f} KB")

Gelöstes Sudoku (extrem, mit Limit):
5 8 6 | 9 4 7 | 3 1 2 
4 3 7 | 2 1 6 | 9 5 8 
9 1 2 | 8 5 3 | 6 7 4 
---------------------
6 2 9 | 5 3 4 | 7 8 1 
1 5 8 | 6 7 2 | 4 3 9 
3 7 4 | 1 8 9 | 5 2 6 
---------------------
8 6 5 | 3 9 1 | 2 4 7 
7 9 1 | 4 2 5 | 8 6 3 
2 4 3 | 7 6 8 | 1 9 5 
Laufzeit: 20.425463 s
Speicher:  8.03 KB


---

# Iterativer DFS (IDS – Iterative Deepening Search)

Das Limit wird schrittweise erhöht, bis eine Lösung gefunden wird.

In [78]:
def solve_ids(board):

    limit = 1
    solved = False

    while not solved:
        current_board = copy.deepcopy(board)
        solved = solve_dfs_limited(current_board, 0, limit)
        limit += 1

    return solved

board = copy.deepcopy(sudoku_diff)
solved, laufzeit, speicher = measure(solve_ids, board)

if solved:
    print("Gelöstes Sudoku (extrem, mit Limit):")
    print_sudoku(solved)
else:
    print("Keine Lösung gefunden")

print(f"Laufzeit: {laufzeit:.6f} s")
print(f"Speicher:  {speicher:.2f} KB")

Gelöstes Sudoku (extrem, mit Limit):
5 8 6 | 9 4 7 | 3 1 2 
4 3 7 | 2 1 6 | 9 5 8 
9 1 2 | 8 5 3 | 6 7 4 
---------------------
6 2 9 | 5 3 4 | 7 8 1 
1 5 8 | 6 7 2 | 4 3 9 
3 7 4 | 1 8 9 | 5 2 6 
---------------------
8 6 5 | 3 9 1 | 2 4 7 
7 9 1 | 4 2 5 | 8 6 3 
2 4 3 | 7 6 8 | 1 9 5 
Laufzeit: 1392.190849 s
Speicher:  10.80 KB


---

# Breadth-First Search (BFS)

## BFS auf einfachem Sudoku

In [79]:
from collections import deque

def solve_bfs(board_bfs):
    queue = deque()
    queue.append(board_bfs)
    while queue:
        first_in_queue = queue.popleft()
        empty = find_empty(first_in_queue)

        # Kein leeres Feld mehr => Sudoku gelöst
        if empty is None:
            return first_in_queue

        for num in range(1, 10):
            curr = copy.deepcopy(first_in_queue)
            if is_valid(curr, num, empty[0], empty[1]):
                curr[empty[0]][empty[1]] = num
                queue.append(curr)

    return False

board = copy.deepcopy(sudoku)
solved, laufzeit, speicher = measure(solve_bfs, board)

if solved:
    print("Gelöstes Sudoku (einfach):")
    print_sudoku(solved)
else:
    print("Keine Lösung gefunden")

print(f"Laufzeit: {laufzeit:.6f} s")
print(f"Speicher:  {speicher:.2f} KB")

Gelöstes Sudoku (einfach):
4 9 6 | 7 1 5 | 8 3 2 
7 5 3 | 4 2 8 | 1 9 6 
2 1 8 | 6 3 9 | 7 4 5 
---------------------
5 3 1 | 2 6 7 | 9 8 4 
6 4 9 | 1 8 3 | 2 5 7 
8 2 7 | 9 5 4 | 6 1 3 
---------------------
3 7 4 | 8 9 2 | 5 6 1 
1 8 5 | 3 7 6 | 4 2 9 
9 6 2 | 5 4 1 | 3 7 8 
Laufzeit: 0.080230 s
Speicher:  24.66 KB


# BFS auf extremem Sudoku

In [80]:
board = copy.deepcopy(sudoku_diff)
solved, laufzeit, speicher = measure(solve_bfs, board)

if solved:
    print("Gelöstes Sudoku (extrem):")
    print_sudoku(solved)
else:
    print("Keine Lösung gefunden")

print(f"Laufzeit: {laufzeit:.6f} s")
print(f"Speicher:  {speicher:.2f} KB")

Gelöstes Sudoku (einfach):
5 8 6 | 9 4 7 | 3 1 2 
4 3 7 | 2 1 6 | 9 5 8 
9 1 2 | 8 5 3 | 6 7 4 
---------------------
6 2 9 | 5 3 4 | 7 8 1 
1 5 8 | 6 7 2 | 4 3 9 
3 7 4 | 1 8 9 | 5 2 6 
---------------------
8 6 5 | 3 9 1 | 2 4 7 
7 9 1 | 4 2 5 | 8 6 3 
2 4 3 | 7 6 8 | 1 9 5 
Laufzeit: 1133.755384 s
Speicher:  361038.01 KB


---

# Informierte Suche

## DFS mit MRV(Minimum Remaining Values) Heuristik

Der effektivste Algorithmus, da er mit einer Heuristik arbeitet, die als nächstes freies Feld immer das wählt, bei dem die wenigsten Zahlen infrage kommen. Das führt zu einer Bedeutenden Verbesserung in der zeitlichen Performance, wobei der Speicherverbrauch aber ähnlich bleibt.

In [81]:
def solve_dfs_mrv(board):
    empty = find_empty_mrv(board)

    # Kein leeres Feld mehr → Sudoku gelöst
    if empty is None:
        return board

    row, col = empty

    # Probiere Zahlen von 1 bis 9
    for num in range(1, 10):
        if is_valid(board, num, row, col):
            board[row][col] = num

            if solve_dfs_mrv(board):
                return board

            board[row][col] = 0  # Backtrack

    return False


board = copy.deepcopy(sudoku_diff)
solved, laufzeit, speicher = measure(solve_dfs_mrv, board)

if solved:
    print("Gelöstes Sudoku (extrem):")
    print_sudoku(solved)
else:
    print("Keine Lösung gefunden")

print(f"Laufzeit: {laufzeit:.6f} s")
print(f"Speicher:  {speicher:.2f} KB")

Gelöstes Sudoku (einfach):
5 8 6 | 9 4 7 | 3 1 2 
4 3 7 | 2 1 6 | 9 5 8 
9 1 2 | 8 5 3 | 6 7 4 
---------------------
6 2 9 | 5 3 4 | 7 8 1 
1 5 8 | 6 7 2 | 4 3 9 
3 7 4 | 1 8 9 | 5 2 6 
---------------------
8 6 5 | 3 9 1 | 2 4 7 
7 9 1 | 4 2 5 | 8 6 3 
2 4 3 | 7 6 8 | 1 9 5 
Laufzeit: 0.681766 s
Speicher:  8.34 KB
